In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
import pymysql
from datetime import datetime,timedelta
import matplotlib.pyplot as plt
import psycopg2
import json
import urllib



db = client.get_database('telematics_db')


db2 = client2.get_database('telematics_db')

getdate=lambda x: pd.datetime.fromtimestamp(x).date()
gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)

In [ ]:
def latest_loc_ping(l):
    vid=l[0]
    day_start=l[1]
    try:
        url=GPS_API_BASE+"/gps/top?vehicleIds={}&fromTime={}&timeCondition=lte".format(int(vid),int(day_start))
       # print(url)
        df_data=json.load(urllib.request.urlopen(url))
        df=df_data['data'][str(int(vid))]
    except Exception as e:
        print(e)
        
    return [df['vehicleId'],df['created'],df['location'],'df']


In [ ]:
def loc_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]

    try:
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
       # print(url)
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(int(vid))])
        #print("LOC PINGS A GAYI")
        
    except Exception as e:
        print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df

In [ ]:
def hb_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:        
        url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
        print(url)
        hb_data=json.load(urllib.request.urlopen(url))
        hb=pd.DataFrame(hb_data['data'][str(int(vid))])
        #print("HB AAYA")

    except Exception as e:

        print("hb pings issue >> " + str(e))
        hb=pd.DataFrame()
        
    return hb


In [ ]:
data=hb_pings([100002,1592822145,1592822145+60*10])

In [ ]:
data

In [ ]:
data['dattime']=data.created.apply(pd.datetime.fromtimestamp)

In [ ]:
data.created=data.created/1000

In [ ]:
data[['vehicleId','fuelOn','engineOn','chargeOn','created','dattime']].sort_values('dattime')

In [ ]:
def latest_loc_ping_all_parameters(l):
    vid=l[0]
    day_start=l[1]
    try:
        url=GPS_API_BASE+"/gps/top?vehicleIds={}&fromTime={}&timeCondition=lte".format(int(vid),int(day_start))
       # print(url)
        df_data=json.load(urllib.request.urlopen(url))
        df=df_data['data'][str(int(vid))]
    except Exception as e:
        print(e)
        
    return df


In [ ]:
a=loc_pings([100002,1592392800,1592392800+60*20])

In [ ]:
a[['time','ignOn','fuelOn','chargeOn']].sort_values('time')[50:]

In [ ]:
while 1>0:
    tm.sleep(2)
    latest_loc_ping_all_parameters([100002,tm.time()])
    


In [ ]:
def latest_hb_ping(l):
    vid=l[0]
    day_start=l[1]
    try:        
        url=GPS_API_BASE+"/heartbeat/top?vehicleIds={}&fromTime={}&timeCondition=lte".format(int(vid),int(day_start))
        hb_data=json.load(urllib.request.urlopen(url))
        hb=hb_data['data'][str(int(vid))]
        #print("HB AAYA")

    except Exception as e:
        print(e)
        
    return [hb['vehicleId'],hb['created'],'hb']


In [ ]:
a=pd.read_csv("HDB_financer_data_3_03_2020 - Vehicle IDs.csv")

In [ ]:
a.head(1)

In [ ]:
def latest_ping_time_and_location(vid):
    loc=latest_loc_ping([vid,tm.time()])
    hb=latest_hb_ping([vid,tm.time()])
    if loc[1]>hb[1]:
        latest_ping_time=loc[1]
    else:
        latest_ping_time=hb[1]
    print([vid,latest_ping_time,loc[2]])
    
    return [vid,latest_ping_time,loc[2]]


In [ ]:
#latest_ping_time_and_location(100002)
latest_hb_ping([100002,tm.time()])

In [ ]:
import multiprocessing 
pool=multiprocessing.Pool()
results=pool.map(latest_ping_time_and_location,a['vehicle_id'][0:10])
pool.close()
pool.join()

In [ ]:
results